## test_snowflake_connection

In [1]:
import json
import logging
import os
from datetime import datetime, timezone
from pathlib import Path

import snowflake.connector
from snowflake.connector import DictCursor
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)

# Verify all required env vars are present before attempting connection
required = [
    "SNOWFLAKE_ACCOUNT",
    "SNOWFLAKE_USER",
    "SNOWFLAKE_PASSWORD",
    "SNOWFLAKE_ROLE",
    "SNOWFLAKE_WAREHOUSE",
    "SNOWFLAKE_DATABASE",
]
missing = [k for k in required if not os.getenv(k)]
if missing:
    print(f"⚠️  Missing env vars: {missing}")
else:
    print("✅ All Snowflake env vars present.")

✅ All Snowflake env vars present.


In [2]:
def get_connection():
    return snowflake.connector.connect(
        account=os.getenv("SNOWFLAKE_ACCOUNT"),
        user=os.getenv("SNOWFLAKE_USER"),
        password=os.getenv("SNOWFLAKE_PASSWORD"),
        role=os.getenv("SNOWFLAKE_ROLE"),
        warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
        database=os.getenv("SNOWFLAKE_DATABASE"),
    )

try:
    conn = get_connection()
    cur  = conn.cursor()
    cur.execute("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()")
    row = cur.fetchone()
    print(f"✅ Connected successfully!")
    print(f"   User      : {row[0]}")
    print(f"   Role      : {row[1]}")
    print(f"   Warehouse : {row[2]}")
    print(f"   Database  : {row[3]}")
    cur.close()
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")

2026-05-25 15:00:03,201 [INFO] Snowflake Connector for Python Version: 3.18.0, Python Version: 3.11.5, Platform: macOS-15.7.4-arm64-arm-64bit
2026-05-25 15:00:03,202 [INFO] Connecting to GLOBAL Snowflake domain
2026-05-25 15:00:03,320 [INFO] Found credentials in shared credentials file: ~/.aws/credentials


✅ Connected successfully!
   User      : VANBRANTLEY
   Role      : SYSADMIN
   Warehouse : NYC_JOB_TRACKER_WH
   Database  : RAW


In [3]:
tables_to_check = [
    ("JSEARCH",     "SRC_POSTINGS"),
    ("THEIRSTACK",  "SRC_POSTINGS"),
    ("BUILTIN",     "SRC_POSTINGS"),
]

try:
    conn = get_connection()
    cur  = conn.cursor()

    print(f"{'Schema':<15} {'Table':<20} {'Status'}")
    print("-" * 50)

    for schema, table in tables_to_check:
        cur.execute(f"""
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_schema = '{schema}'
            AND   table_name   = '{table}'
        """)
        exists = cur.fetchone()[0] > 0
        status = "✅ exists" if exists else "❌ NOT FOUND"
        print(f"  {schema:<13} {table:<20} {status}")

    cur.close()
    conn.close()
except Exception as e:
    print(f"❌ Error: {e}")

2026-05-25 15:01:48,730 [INFO] Snowflake Connector for Python Version: 3.18.0, Python Version: 3.11.5, Platform: macOS-15.7.4-arm64-arm-64bit
2026-05-25 15:01:48,732 [INFO] Connecting to GLOBAL Snowflake domain


Schema          Table                Status
--------------------------------------------------
  JSEARCH       SRC_POSTINGS         ✅ exists
  THEIRSTACK    SRC_POSTINGS         ✅ exists
  BUILTIN       SRC_POSTINGS         ✅ exists


In [4]:
CACHE_DIR = Path("./cache")

# Load one row from each source cache
test_rows = []

# JSearch
jsearch_cache = CACHE_DIR / "jsearch_raw_response.json"
if jsearch_cache.exists():
    with jsearch_cache.open() as f:
        data = json.load(f)
    jobs = data.get("data", {}).get("jobs", [])
    if jobs:
        test_rows.append({
            "SOURCE":      "jsearch:Data Analyst in New York",
            "RAW_PAYLOAD": jobs[0],
            "INGESTED_AT": datetime.now(timezone.utc).isoformat(),
        })
        print(f"✅ JSearch: 1 test row loaded from cache.")
else:
    print("⚠️  JSearch cache not found.")

# TheirStack
theirstack_cache = CACHE_DIR / "theirstack_paid_response.json"
if theirstack_cache.exists():
    with theirstack_cache.open() as f:
        data = json.load(f)
    jobs = data.get("data", [])
    if jobs:
        test_rows.append({
            "SOURCE":      "theirstack:nyc-data-roles",
            "RAW_PAYLOAD": jobs[0],
            "INGESTED_AT": datetime.now(timezone.utc).isoformat(),
        })
        print(f"✅ TheirStack: 1 test row loaded from cache.")
else:
    print("⚠️  TheirStack cache not found.")

# Built In NYC
builtin_cache = CACHE_DIR / "builtin_scraped_jobs.json"
if builtin_cache.exists():
    with builtin_cache.open() as f:
        scraped = json.load(f)
    if scraped:
        record = scraped[0]
        test_rows.append({
            "SOURCE":      "builtin_nyc",
            "RAW_PAYLOAD": {
                "source_url":  record["source_url"],
                "crawl_title": record["crawl_title"],
                "scraped_at":  record["scraped_at"],
                **record["job_posting"],
            },
            "INGESTED_AT": datetime.now(timezone.utc).isoformat(),
        })
        print(f"✅ Built In NYC: 1 test row loaded from cache.")
else:
    print("⚠️  Built In NYC cache not found.")

print(f"\nTotal test rows ready: {len(test_rows)}")

✅ JSearch: 1 test row loaded from cache.
✅ TheirStack: 1 test row loaded from cache.
✅ Built In NYC: 1 test row loaded from cache.

Total test rows ready: 3


In [5]:
# Maps the start of a SOURCE string to its target table
SOURCE_TABLE_MAP = {
    "jsearch":     "RAW.JSEARCH.SRC_POSTINGS",
    "theirstack":  "RAW.THEIRSTACK.SRC_POSTINGS",
    "builtin_nyc": "RAW.BUILTIN.SRC_POSTINGS",
}

def get_target_table(source: str) -> str:
    for prefix, table in SOURCE_TABLE_MAP.items():
        if source.startswith(prefix):
            return table
    raise ValueError(f"Unknown source prefix: {source!r}")

try:
    conn = get_connection()
    cur  = conn.cursor()

    for row in test_rows:
        table        = get_target_table(row["SOURCE"])
        raw_payload  = json.dumps(row["RAW_PAYLOAD"], default=str)
        ingested_at  = row["INGESTED_AT"]
        source       = row["SOURCE"]

        cur.execute(
            f"""
            INSERT INTO {table} (SOURCE, RAW_PAYLOAD, INGESTED_AT)
            SELECT %s, PARSE_JSON(%s), %s::TIMESTAMP_TZ
            """,
            (source, raw_payload, ingested_at)
        )
        print(f"✅ Inserted 1 row into {table}")

    conn.commit()
    cur.close()
    conn.close()
    print("\n✅ All test rows committed.")

except Exception as e:
    print(f"❌ Insert failed: {e}")

2026-05-25 15:08:17,788 [INFO] Snowflake Connector for Python Version: 3.18.0, Python Version: 3.11.5, Platform: macOS-15.7.4-arm64-arm-64bit
2026-05-25 15:08:17,792 [INFO] Connecting to GLOBAL Snowflake domain


✅ Inserted 1 row into RAW.JSEARCH.SRC_POSTINGS
✅ Inserted 1 row into RAW.THEIRSTACK.SRC_POSTINGS
✅ Inserted 1 row into RAW.BUILTIN.SRC_POSTINGS

✅ All test rows committed.


In [6]:
try:
    conn = get_connection()
    cur  = conn.cursor(DictCursor)

    for schema in ["JSEARCH", "THEIRSTACK", "BUILTIN"]:
        cur.execute(f"""
            SELECT
                SOURCE,
                INGESTED_AT,
                RAW_PAYLOAD
            FROM RAW.{schema}.SRC_POSTINGS
            ORDER BY INGESTED_AT DESC
            LIMIT 1
        """)
        row = cur.fetchone()
        if row:
            # Truncate RAW_PAYLOAD for display
            payload_preview = str(row["RAW_PAYLOAD"])[:120] + "..."
            print(f"✅ RAW.{schema}.SRC_POSTINGS")
            print(f"   SOURCE      : {row['SOURCE']}")
            print(f"   INGESTED_AT : {row['INGESTED_AT']}")
            print(f"   RAW_PAYLOAD : {payload_preview}")
            print()
        else:
            print(f"⚠️  RAW.{schema}.SRC_POSTINGS — no rows found.")

    cur.close()
    conn.close()
except Exception as e:
    print(f"❌ Verification failed: {e}")

2026-05-25 15:08:29,812 [INFO] Snowflake Connector for Python Version: 3.18.0, Python Version: 3.11.5, Platform: macOS-15.7.4-arm64-arm-64bit
2026-05-25 15:08:29,814 [INFO] Connecting to GLOBAL Snowflake domain


✅ RAW.JSEARCH.SRC_POSTINGS
   SOURCE      : jsearch:Data Analyst in New York
   INGESTED_AT : 2026-05-25 19:04:42.599092+00:00
   RAW_PAYLOAD : {
  "apply_options": [
    {
      "apply_link": "https://www.linkedin.com/jobs/view/data-analyst-new-york-at-jobright-a...

✅ RAW.THEIRSTACK.SRC_POSTINGS
   SOURCE      : theirstack:nyc-data-roles
   INGESTED_AT : 2026-05-25 19:04:42.607133+00:00
   RAW_PAYLOAD : {
  "avg_annual_salary_usd": null,
  "cities": [],
  "closed_at": null,
  "company": "Aisle and Abroad",
  "company_doma...

✅ RAW.BUILTIN.SRC_POSTINGS
   SOURCE      : builtin_nyc
   INGESTED_AT : 2026-05-25 19:04:42.612207+00:00
   RAW_PAYLOAD : {
  "@context": "https://schema.org",
  "@type": "JobPosting",
  "applicantLocationRequirements": {
    "@type": "Countr...

